# CNN And ResNet Explainer

## Simple English First
A convolutional neural network, or CNN, learns visual patterns from images. In TyreVisionX, CNN-based models are used to look for patterns that may suggest a tyre defect.

This notebook explains two model families used in the repo:
- a small baseline CNN for fast experiments
- ResNet backbones for stronger image classification


## CNN Basics
### Simple explanation
A CNN looks at small image regions, learns useful patterns, and combines them into a final prediction.

### Technical detail
Convolutions apply learnable filters across the image. Pooling reduces spatial size while preserving strong signals. Stacking these layers allows the network to move from simple edges to higher-level texture and structural patterns.

## Why CNNs Matter In TyreVisionX
### Simple explanation
Tyre defects often show up as texture disruptions, cracks, unusual edges, or small local changes. CNNs are a natural first tool for this kind of image problem.

### Technical detail
CNNs can capture tread texture, groove layout, sidewall transitions, and local defect-like contrast changes. That makes them suitable for the current binary classification stage.

## SimpleCNN In This Repository
### Simple explanation
The SimpleCNN path is the small, fast baseline. It is useful for early experiments and for testing ideas quickly.

### Technical detail
The historical baseline path in the repo includes a small CNN and several regularization experiments. This path is now archived under `src/legacy/` for reproducibility, not treated as the main current pipeline.

In [1]:
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the TyreVisionX repo root from the current working directory.')

repo_root = find_repo_root()
print('Repo root:', repo_root)
simple_cnn_path = repo_root / 'src/models/simple_cnn.py'
text = simple_cnn_path.read_text(encoding='utf-8').splitlines()
for line in text[:80]:
    print(line)

Repo root: /Users/ritik/Documents/Project TDA/TyreVisionX
"""Small CNN baseline for tyre defect classification (binary sigmoid output)."""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class SimpleCNN(nn.Module):
    """Phase-1 tiny baseline:

    Input(224x224x3) -> Conv16 -> (BatchNorm) -> ReLU -> MaxPool ->
    Conv32 -> (BatchNorm) -> ReLU -> MaxPool ->
    Flatten -> (Dropout) -> Linear -> (optional Sigmoid)
    """

    def __init__(
        self,
        in_channels: int = 3,
        use_batchnorm: bool = False,
        dropout: float = 0.0,
        output_logits: bool = False,
    ) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16) if use_batchnorm else nn.Identity()
        self.bn2 = nn.BatchNorm2d(32) if use_batchnorm else nn.Identity()
        self.pool = 

## ResNet In This Repository
### Simple explanation
ResNet is a stronger CNN family that uses skip connections. These connections help deeper models learn without falling apart during training.

### Technical detail
TyreVisionX currently uses ResNet-18 and ResNet-34 as the canonical active models. The active training path builds them through a small wrapper and supports an optional GNN extension.

In [2]:
resnet_path = repo_root / 'src/models/resnet_classifier.py'
text = resnet_path.read_text(encoding='utf-8').splitlines()
for line in text[:120]:
    print(line)

"""ResNet classifier builders."""
from __future__ import annotations

from typing import Literal

import torch.nn as nn

from src.utils.torchvision_compat import load_torchvision_models

models = load_torchvision_models()


def build_resnet(model_name: Literal["resnet18", "resnet34"], num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    if model_name == "resnet18":
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.resnet18(weights=weights)
    elif model_name == "resnet34":
        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.resnet34(weights=weights)
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


## Transfer Learning
### Simple explanation
Transfer learning means starting from a model that already learned useful image features from a large dataset.

### Technical detail
The ResNet path uses pretrained ImageNet weights. This is especially useful in early-stage research because it usually improves training efficiency and reduces the amount of task-specific data needed.

## Why Keep Both CNN And ResNet Stories
The professor-facing value is different:
- SimpleCNN shows the low-capacity baseline story.
- ResNet shows the stronger transfer-learning story.
- Comparing them helps frame why the project moved toward the canonical ResNet-based pipeline.

## Future Work Sections
### Anomaly detection
Simple explanation: detect unusual tyres even when labeled defect categories are incomplete.

Technical detail: this may use reconstruction error, embedding distance, or one-class approaches.

### Localization and segmentation
Simple explanation: show where a defect is.

Technical detail: classification can become a stepping stone toward bounding-box or pixel-level supervision.

### Multi-view 3D
Simple explanation: use several views of the same tyre.

Technical detail: this would support better surface reasoning than single-image classification.

### Knowledge reasoning
Simple explanation: connect image evidence with defect knowledge and possible causes.

Technical detail: still roadmap material.